# Jurimetria Preditiva em Acórdãos do TCU — Saúde e Educação
### Trabalho final — Deep Learning e PLN (IDP) · Modalidade 2 (NLP no Setor Público)

**Integrantes:** _(preencher)_  
**Data:** _(preencher)_

---

**Problema.** Predizer o desfecho de acórdãos do TCU relacionados às áreas de Saúde e Educação:
contas **irregulares** (condenação/multa) ou **regulares**.  
Tarefa: **classificação de texto** em português jurídico.

**Fonte de dados:** Portal de Dados Abertos do TCU — CSV oficial, sem scraping.  
Disponível em: https://sites.tcu.gov.br/dados-abertos/jurisprudencia/

**Hipótese central.** Um modelo de Deep Learning (LegalBert-pt com truncação head+tail) supera o baseline clássico (TF-IDF + modelo linear) na métrica **F1-macro**.

> Notebook **orquestrador**: a lógica pesada vive em `src/`; aqui importamos, chamamos e narramos.

## 0. Setup e configuração

In [ ]:
import sys
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

RAIZ = Path.cwd().parent
sys.path.insert(0, str(RAIZ))

DATA_RAW = RAIZ / "data" / "raw"
DATA_INTERIM = RAIZ / "data" / "interim"
DATA_PROCESSED = RAIZ / "data" / "processed"
RESULTADOS = RAIZ / "resultados"

for p in (DATA_RAW, DATA_INTERIM, DATA_PROCESSED, RESULTADOS, RESULTADOS / "figuras"):
    p.mkdir(parents=True, exist_ok=True)

print("Ambiente configurado. Seed =", RANDOM_STATE)
print("Raiz do projeto:", RAIZ)

## 1. Aquisição de dados

Download direto dos CSVs do Portal de Dados Abertos do TCU — **sem scraping**.  
Guardrail 2: stream para disco, sem carregar na RAM durante o download.

In [ ]:
from src.aquisicao.baixar_csvs import baixar_varios

ANOS = [2023, 2024]

arquivos = baixar_varios(ANOS, DATA_RAW)
for a in arquivos:
    print(f"{a.name}  ({a.stat().st_size / 1e6:.0f} MB)")

## 2. Inspeção do CSV — confirmar D-05 e D-06

Verificar colunas reais antes de qualquer modelagem.

In [ ]:
from src.preprocessamento.filtrar_tematico import inspecionar_colunas

arquivo_exemplo = DATA_RAW / f"acordao-completo-{ANOS[-1]}.csv"
colunas = inspecionar_colunas(arquivo_exemplo)
print("Colunas disponíveis:", colunas)

df_amostra = pd.read_csv(arquivo_exemplo, nrows=3, sep=None, engine="python")
df_amostra

## 3. Filtro temático e extração de label

Guardrail 3: filtrar por termos de saúde/educação.  
Resultado esperado: 2.000–4.000 acórdãos.

In [ ]:
from src.preprocessamento.filtrar_tematico import combinar_anos

df = combinar_anos(ANOS, DATA_RAW)

saida_interim = DATA_INTERIM / "acordaos_filtrados.parquet"
df.to_parquet(saida_interim, index=False)

print(f"Corpus filtrado: {len(df)} acórdãos")
print(f"Memória: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print("\nDistribuição de labels:")
print(df["label"].value_counts())

## 4. Análise Exploratória de Dados (EDA)

In [ ]:
df = pd.read_parquet(DATA_INTERIM / "acordaos_filtrados.parquet")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

df["label"].value_counts().plot(kind="bar", ax=axes[0], color=["#d62728", "#2ca02c", "#1f77b4"])
axes[0].set_title("Distribuição de Classes")
axes[0].tick_params(axis="x", rotation=30)

df["n_palavras"] = df["sumario"].fillna("").str.split().str.len()
df["n_palavras"].hist(bins=50, ax=axes[1], color="steelblue")
axes[1].set_title("Tamanho do Sumário (palavras)")
axes[1].axvline(df["n_palavras"].median(), color="red", linestyle="--",
                label=f'Mediana={df["n_palavras"].median():.0f}')
axes[1].legend()

if "anoAcordao" in df.columns:
    df.groupby(["anoAcordao", "label"]).size().unstack().plot(kind="bar", ax=axes[2])
    axes[2].set_title("Acórdãos por Ano e Desfecho")
    axes[2].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig(RESULTADOS / "figuras" / "eda_visao_geral.png", dpi=150)
plt.show()
print(df["n_palavras"].describe())

## 5. Pré-processamento e Split

Split estratificado: **70% treino / 15% validação / 15% teste** (seed=42).

In [ ]:
from src.preprocessamento.limpeza import limpar_coluna, dividir_dados

df = pd.read_parquet(DATA_INTERIM / "acordaos_filtrados.parquet")

df = limpar_coluna(df, coluna="sumario", modo="bert")
df = df.rename(columns={"texto_limpo": "texto_bert"})

df = limpar_coluna(df, coluna="sumario", modo="tfidf")
df = df.rename(columns={"texto_limpo": "texto_tfidf"})

X_train_bert, X_val_bert, X_test_bert, y_train, y_val, y_test = dividir_dados(
    df, coluna_texto="texto_bert", coluna_label="label", seed=RANDOM_STATE
)

X_train_tfidf = df.loc[X_train_bert.index, "texto_tfidf"]
X_val_tfidf   = df.loc[X_val_bert.index,   "texto_tfidf"]
X_test_tfidf  = df.loc[X_test_bert.index,  "texto_tfidf"]

print(f"Treino: {len(X_train_bert)} | Val: {len(X_val_bert)} | Teste: {len(X_test_bert)}")
print(y_train.value_counts())

df.to_parquet(DATA_PROCESSED / "dados_processados.parquet", index=False)

## 6. Baseline — TF-IDF + modelo linear

Piso de performance — o LegalBert-pt precisa superar este F1-macro.

In [ ]:
from src.modelos.baseline import treinar_baseline, avaliar

pipe_lr,  pred_lr,  = treinar_baseline(X_train_tfidf, y_train, X_test_tfidf, modelo="logistic", seed=RANDOM_STATE)[:2]
metricas_lr  = avaliar(y_test, pred_lr,  "TF-IDF + LogisticRegression")

pipe_svm, pred_svm = treinar_baseline(X_train_tfidf, y_train, X_test_tfidf, modelo="svm",     seed=RANDOM_STATE)[:2]
metricas_svm = avaliar(y_test, pred_svm, "TF-IDF + LinearSVC")

if metricas_lr["f1_macro"] >= metricas_svm["f1_macro"]:
    pipe_baseline, pred_baseline, metricas_baseline = pipe_lr, pred_lr, metricas_lr
    nome_baseline = "TF-IDF + LogisticRegression"
else:
    pipe_baseline, pred_baseline, metricas_baseline = pipe_svm, pred_svm, metricas_svm
    nome_baseline = "TF-IDF + LinearSVC"

print(f"\nBaseline: {nome_baseline}  F1-macro = {metricas_baseline['f1_macro']:.4f}")

## 7. Deep Learning — Fine-tuning LegalBert-pt

Modelo: `dominguesm/legal-bert-base-cased-ptbr` com truncação **head+tail** (128+384).  
> Executar no **Google Colab (GPU T4)**.

In [ ]:
from src.modelos.transformer import treinar_transformer
from src.avaliacao.metricas import calcular_metricas

modelo_dl, pred_transformer, encoder_rotulos = treinar_transformer(
    X_train=X_train_bert,
    y_train=y_train,
    X_val=X_val_bert,
    y_val=y_val,
    X_test=X_test_bert,
    modelo_nome="dominguesm/legal-bert-base-cased-ptbr",
    max_head=128,
    max_tail=384,
    epocas=3,
    batch_size=16,
    lr=2e-5,
    seed=RANDOM_STATE,
)

metricas_transformer = calcular_metricas(y_test, pred_transformer, "LegalBert-pt head+tail")

## 8. Avaliação comparativa

In [ ]:
from src.avaliacao.metricas import comparar_modelos, plotar_f1_por_classe

classes = sorted(y_test.unique().tolist())

resultado_final = comparar_modelos(
    y_test=y_test,
    pred_baseline=pred_baseline,
    pred_transformer=pred_transformer,
    classes=classes,
)

plotar_f1_por_classe(y_test, pred_baseline, pred_transformer, classes)
print(f"\nGanho F1-macro: {resultado_final['ganho_f1_macro']:+.4f}")

## 9. Diferencial — Explicabilidade com LIME

Quais termos do acórdão mais predizem **condenação** (contas irregulares)?

In [ ]:
from src.avaliacao.metricas import explicar_com_lime

idx_irregulares = [i for i, p in enumerate(pred_transformer) if p == "Irregular"]
textos_lime = X_test_tfidf.iloc[idx_irregulares[:5]].tolist()

caminho_lime = explicar_com_lime(
    pipeline_baseline=pipe_baseline,
    textos_teste=textos_lime,
    classes=classes,
    num_features=10,
    num_amostras=min(5, len(textos_lime)),
)
print(f"Plot LIME: {caminho_lime}")

## 10. Conclusão

**Resultados:**

| Modelo | F1-macro | Ganho |
|--------|----------|-------|
| TF-IDF + LogReg/SVM (baseline) | _(preencher)_ | — |
| LegalBert-pt head+tail | _(preencher)_ | _(preencher)_ |

**Impacto prático:** O modelo permite que gestores de saúde/educação obtenham um *score* de risco de condenação pelo TCU antes da auditoria, possibilitando correções preventivas.

**Limitações:** corpus limitado a 2 anos; possível desbalanceamento entre classes.

---
Ver referências completas em `docs/referencias.md` (5 domínio + 5 técnica, ABNT).